# Quantum Secret Sharing Simulator — Academic Walkthrough

**Author:** Sumit Tapas Chongder  
**Affiliation:** M.Tech in Quantum Technologies, IIT Jodhpur  
**Supervisor:** Dr. Atul Kumar  
**Course:** Seminal Features of Quantum Information

---

This notebook provides a complete academic walkthrough of the Quantum Secret Sharing (QSS) protocol using GHZ states. It accompanies the interactive Streamlit dashboard.

## Contents
1. GHZ State Construction
2. Protocol Simulation (3-node)
3. Eavesdropper Detection
4. Noise Analysis
5. Tittel et al. (2000) Optical Experiment Replica
6. N-Node Generalisation
7. Approach 1 vs Approach 2 Benchmark

In [ ]:
# Install dependencies if running in Colab
# !pip install qutip qiskit qiskit-aer plotly numpy scipy

import numpy as np
import matplotlib.pyplot as plt

print('NumPy version:', np.__version__)

## 1. GHZ State Construction

The N-qubit GHZ state is defined as:

$$|GHZ_N\rangle = \frac{1}{\sqrt{2}}(|00\cdots0\rangle + |11\cdots1\rangle)$$

We build it using QuTiP tensor products.

In [ ]:
from qutip import basis, tensor, ket2dm, ptrace, entropy_vn

def build_ghz(n):
    """Build N-qubit GHZ state."""
    zero = basis(2, 0)
    one  = basis(2, 1)
    return (tensor([zero]*n) + tensor([one]*n)).unit()

# Build and verify
ghz3 = build_ghz(3)
print(f'GHZ(3) norm: {ghz3.norm():.10f}  (should be 1.0)')
print(f'GHZ(3) shape: {ghz3.shape}')

# Verify entanglement: reduced state of qubit 0 should be maximally mixed
dm3 = ket2dm(ghz3)
rho_A = ptrace(dm3, 0)
entropy = float(entropy_vn(rho_A, base=2))
print(f'Subsystem entropy of qubit A: {entropy:.6f}  (should be 1.0 for max entanglement)')

In [ ]:
# Verify for all N from 3 to 7
print(f'{'N':>3}  {'Norm':>12}  {'Subsystem Entropy':>18}  {'Dim':>6}')
print('-' * 45)
for n in range(3, 8):
    state = build_ghz(n)
    dm = ket2dm(state)
    rho_sub = ptrace(dm, 0)
    ent = float(entropy_vn(rho_sub, base=2))
    print(f'{n:>3}  {state.norm():>12.10f}  {ent:>18.6f}  {2**n:>6}')

## 2. Protocol Simulation — 3-node case

### Phase gate

$$P(\phi) = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\phi} \end{pmatrix}$$

### Parity condition

The protocol proceeds only if $\alpha + \beta + \gamma \in \{0, \pi\}$.

### Secret reconstruction

$$c = a \cdot b \cdot d \quad \text{where } d = \cos(\alpha+\beta+\gamma) = \pm 1$$

In [ ]:
from qss.protocol import parity_check, reconstruct_secret, run_protocol_round, PHASES

# Test all valid phase combinations
valid_combos = [(0,0,0), (1,1,0), (1,0,1), (0,1,1)]
invalid_combos = [(1,0,0), (0,1,0), (0,0,1), (1,1,1)]

print('Valid phase combinations (parity != 0):')
for ai, bi, gi in valid_combos:
    p = parity_check(ai, bi, gi)
    phases_str = f'{PHASES[ai]:.4f} + {PHASES[bi]:.4f} + {PHASES[gi]:.4f} = {PHASES[ai]+PHASES[bi]+PHASES[gi]:.4f}'
    print(f'  ({ai},{bi},{gi}): {phases_str} rad  →  parity = {p:+d}')

print()
print('Invalid combinations (discard):')
for ai, bi, gi in invalid_combos:
    p = parity_check(ai, bi, gi)
    print(f'  ({ai},{bi},{gi}): parity = {p} (discard)')

In [ ]:
# Run full protocol rounds
print(f'{'Phases':>15}  {'Parity':>8}  {'Outcomes':>15}  {'Secret':>8}  {'Inferred':>10}  {'Success':>8}')
print('-' * 75)

for phase_indices in valid_combos:
    result = run_protocol_round(3, phase_indices, seed=42)
    phases_str = str([f'{PHASES[i]:.2f}' for i in phase_indices])
    print(f'{phases_str:>15}  {result["parity"]:>+8d}  '
          f'{str(result["outcomes"]):>15}  {result["secret_value"]:>+8d}  '
          f'{result["inferred_value"]:>+10d}  {str(result["success"]):>8}')

## 3. Eavesdropper Detection

Theoretical QBER for intercept-resend attack:

$$\text{QBER} = \frac{p_{\text{intercept}}}{2}$$

Detection threshold: QBER > 5%

In [ ]:
from qss.eavesdropper import inject_eve_attack, qber_vs_intercept_curve, detection_probability

# QBER vs intercept probability
curve = qber_vs_intercept_curve(200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: QBER curve
ax = axes[0]
ax.plot(curve[:, 0], curve[:, 1], 'b-', linewidth=2, label='QBER = p/2')
ax.axhline(0.05, color='red', linestyle='--', label='5% threshold')
ax.fill_between(curve[:, 0], 0.05, curve[:, 1],
                where=curve[:, 1] > 0.05, alpha=0.2, color='red', label='Eve detected')
ax.set_xlabel('Eve intercept probability p')
ax.set_ylabel('QBER')
ax.set_title('QBER vs Eve intercept probability')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Detection probability vs rounds
ax2 = axes[1]
rounds = np.logspace(1, 4, 100)
for p, color in [(0.1, 'blue'), (0.3, 'orange'), (0.5, 'red'), (1.0, 'purple')]:
    det_probs = [detection_probability(p, int(r)) for r in rounds]
    ax2.semilogx(rounds, det_probs, color=color, linewidth=2, label=f'p={p}')
ax2.axhline(0.99, color='gray', linestyle=':', alpha=0.7)
ax2.set_xlabel('Number of rounds (log scale)')
ax2.set_ylabel('P(Eve detected)')
ax2.set_title('Detection probability vs rounds')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/eve_detection.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved to docs/eve_detection.png')

## 4. Noise Analysis

**Depolarising fidelity:** $F(p) = (1-p)^N + p^N/2^N$

**Dephasing fidelity:** $F(p) = (1-p)^N$

In [ ]:
from qss.noise import depolarise_sweep, dephase_sweep, threshold_noise

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for ax, model_name, sweep_fn in [
    (axes[0], 'Depolarising', depolarise_sweep),
    (axes[1], 'Dephasing', dephase_sweep),
]:
    for i, n in enumerate(range(3, 9)):
        curve = sweep_fn(n, 0.5)
        ax.plot(curve[:, 0], curve[:, 1],
                color=colors[i], linewidth=2, label=f'N={n}')
    ax.axhline(0.9, color='gray', linestyle='--', alpha=0.7, label='F=0.9 threshold')
    ax.set_xlabel('Noise probability p')
    ax.set_ylabel('Fidelity F')
    ax.set_title(f'GHZ fidelity — {model_name} noise')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../docs/noise_fidelity.png', dpi=120, bbox_inches='tight')
plt.show()

# Noise thresholds
print(f'\nNoise threshold for F > 0.9:')
print(f'{'N':>3}  {'Depolarise':>12}  {'Dephase':>10}')
print('-' * 30)
for n in range(3, 9):
    td = threshold_noise(n, 'depolarise', 0.9)
    tph = threshold_noise(n, 'dephase', 0.9)
    print(f'{n:>3}  {td:>12.4f}  {tph:>10.4f}')

## 5. Tittel et al. (2000) — Optical Experiment Replica

$$P_{ijk} = \frac{1}{8}\bigl[1 + ijk\cos(\alpha+\beta+\gamma)\bigr]$$

In [ ]:
from qss.tittel2000 import all_outcome_probabilities, rate_vs_distance, max_range_km

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: Outcome probabilities at two phase settings
ax = axes[0]
for alpha, beta, gamma, label, color in [
    (0, 0, 0, 'α+β+γ=0 (even parity)', '#2A9D8F'),
    (np.pi/2, np.pi/2, 0, 'α+β+γ=π (odd parity)', '#E76F51'),
]:
    probs = all_outcome_probabilities(alpha, beta, gamma)
    labels_list = list(probs.keys())
    values = list(probs.values())
    x = np.arange(len(labels_list))
    ax.bar(x + (0.2 if 'even' in label else -0.2), values, width=0.35,
           label=label, color=color, alpha=0.8)

ax.set_xticks(range(8))
ax.set_xticklabels(list(all_outcome_probabilities(0, 0, 0).keys()),
                   rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Probability')
ax.set_title('P_ijk outcomes — Tittel 2000')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Right: Generation rate vs distance
ax2 = axes[1]
curve = rate_vs_distance(60, 300)
ax2.plot(curve[:, 0], curve[:, 1], 'purple', linewidth=2)
ax2.axhline(0.1, color='red', linestyle='--', label='Min usable rate (0.1 Hz)')
ax2.axvline(max_range_km(), color='orange', linestyle=':', label=f'Max range ({max_range_km():.1f} km)')
ax2.set_xlabel('Fibre distance (km)')
ax2.set_ylabel('Generation rate (Hz)')
ax2.set_title('Secret bit rate vs distance')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/tittel2000.png', dpi=120, bbox_inches='tight')
plt.show()

# Verify probability sum
for alpha, beta, gamma in [(0,0,0), (np.pi/2, np.pi/2, 0), (1.0, 0.5, 0.3)]:
    probs = all_outcome_probabilities(alpha, beta, gamma)
    total = sum(probs.values())
    print(f'α={alpha:.2f}, β={beta:.2f}, γ={gamma:.2f}: Σ P_ijk = {total:.12f}')

## 6. N-Node Generalisation

In [ ]:
from qss.noise import hilbert_space_dim, memory_mb_estimate

print(f'{'N':>3}  {'Dim 2^N':>8}  {'DM RAM (MB)':>12}  {'Depol threshold':>16}  {'Dephase threshold':>18}')
print('-' * 65)
for n in range(3, 9):
    dim = hilbert_space_dim(n)
    mem = memory_mb_estimate(n)
    td = threshold_noise(n, 'depolarise', 0.9)
    tph = threshold_noise(n, 'dephase', 0.9)
    warn = ' ⚠ large' if mem > 100 else ''
    print(f'{n:>3}  {dim:>8}  {mem:>12.3f}  {td:>16.4f}  {tph:>18.4f}{warn}')

## 7. Approach 1 vs Approach 2 Benchmark

In [ ]:
from qss.ghz import compare_approaches

ns = list(range(3, 9))
q1 = [compare_approaches(n)['approach1']['qubit_count'] for n in ns]
q2 = [compare_approaches(n)['approach2']['qubit_count'] for n in ns]
m1 = [compare_approaches(n)['approach1']['msg_count'] for n in ns]
m2 = [compare_approaches(n)['approach2']['msg_count'] for n in ns]
f1 = [compare_approaches(n)['approach1']['fidelity_estimate'] for n in ns]
f2 = [compare_approaches(n)['approach2']['fidelity_estimate'] for n in ns]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, y1, y2, ylabel, title in [
    (axes[0], q1, q2, 'Qubit count', 'Qubit count vs N'),
    (axes[1], m1, m2, 'Classical messages', 'Message overhead vs N'),
    (axes[2], f1, f2, 'Fidelity estimate', 'Fidelity estimate vs N'),
]:
    ax.plot(ns, y1, 'o-', color='#E76F51', linewidth=2, markersize=7, label='Approach 1')
    ax.plot(ns, y2, 's-', color='#2A9D8F', linewidth=2, markersize=7, label='Approach 2')
    ax.set_xlabel('Number of nodes N')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/benchmark.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nSummary table (N=5):')
data = compare_approaches(5)
for approach, info in data.items():
    print(f'  {approach}:')
    for k, v in info.items():
        print(f'    {k}: {v}')

## Summary

This notebook has demonstrated:

1. **GHZ state** construction and verification (norm = 1, subsystem entropy = 1 bit)
2. **QSS protocol** — parity check, measurement simulation, secret reconstruction
3. **Eavesdropper detection** — QBER = p/2 theoretical, threshold at 5%
4. **Noise models** — fidelity degrades exponentially with N
5. **Tittel 2000** — P_ijk formula verified, detection probabilities sum to 1
6. **N-node scaling** — Hilbert space grows as 2^N
7. **Approach 2 superiority** — fewer qubits, fewer messages, higher fidelity

---

For the interactive dashboard, run:

```bash
streamlit run ../app.py
```

Or visit the live app at [sumitchongder-quantum-secret-sharing-simulator.streamlit.app](https://sumitchongder-quantum-secret-sharing-simulator.streamlit.app)